
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Expanded-Universe Final Multi-Horizon Experiment

This notebook freezes the final finance retrieval configuration and evaluates reproducibility across multiple forecast horizons on the expanded stock universe:

\[
\boxed{H\in\{1,5,20\}}.
\]

---

# Final retrieval configuration

Past length: \(L=20\)  
Retrieved neighbors: \(K=10\)  
Pattern candidate pool: \(M=100\)  
Learned context: **stock-local context only**

Local features:

- STK_RET_5
- STK_RET_20
- STK_RV_20
- STK_VOL_RATIO
- STK_DD_60
- REL_STRENGTH_20

Seeds: \(\{0,1,2\}\).

---

# Methods compared at every horizon

- **SameStock Pattern**: Pattern retrieval from the same stock's historical memory.
- **SameStock Learned**: Future-compatible reranking of the same-stock Pattern Top-100 pool.
- **CrossStock Pattern**: Pattern retrieval from the full stock universe.
- **CrossStock Learned**: Future-compatible reranking of the full-universe Pattern Top-100 pool.

The primary comparisons are Pattern -> Future-Compatible Learned and Same-stock memory -> Cross-stock memory.

---

# Horizon-specific target

For each horizon the target is the future cumulative log-return path,

\[
y^{(H)}=[r_{t+1},\ r_{t+1}+r_{t+2},\ \ldots,\ \sum_{j=1}^{H}r_{t+j}].
\]

Thus \(H=1,5,20\) correspond to one-day, one-week-scale, and approximately monthly paths.

---

# Common forecast-origin design

The H=5 expanded-window table supplies the ticker, anchor date, pattern, observable context, and sector. Raw adjusted prices are then used to reconstruct H=1/5/20 futures at the same anchors, so changing the horizon does not alter the observed past/context. Only anchors without a complete H-step future are excluded.

---

# Temporal protocol

For each horizon, admissibility is based on `FutureEndDate`:

- Train memory: future end <= 2009-12-31
- Train queries: 2010--2014
- Validation memory: future end <= 2014-12-31
- Validation queries: 2015--2019
- Test memory: future end <= 2019-12-31
- Test queries: anchor date >= 2020

This prevents future leakage.

---

# Main paper claims tested

1. Future-compatibility learning improves Pattern retrieval across horizons.
2. Cross-security memory can provide additional gain beyond same-security memory.
3. The result is not specific to the five-day horizon.

Primary metrics are AnalogFutureMSE, ForecastMSE, and TerminalMAE. Direction Accuracy is secondary. Statistical comparisons use a date-aware moving-block bootstrap with 5,000 repetitions.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


In [ ]:

from pathlib import Path
import math
import random
import warnings
import gc
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_ROOT = REPO_WORK_ROOT / "finance_case"

BASE_RESULT_DIR = (
    DATA_ROOT /
    "results" /
    "yahoo_crossstock_ablation"
)

RAW_FILE = (
    DATA_ROOT /
    "raw" /
    "yahoo_sp500_current_2000_2025.parquet"
)

BASE_WINDOW_FILE = (
    BASE_RESULT_DIR /
    "00_expanded_windows.parquet"
)

RESULT_DIR = (
    DATA_ROOT /
    "results" /
    "expanded_multihorizon_final"
)

CACHE_DIR = (
    RESULT_DIR /
    "cache"
)

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert RAW_FILE.exists(), RAW_FILE
assert BASE_WINDOW_FILE.exists(), BASE_WINDOW_FILE

HORIZONS = [1, 5, 20]

SEEDS = [0, 1, 2]

TOP_K = 10
TOP_M = 100

TRAIN_BATCH = 128
EVAL_BATCH = 256
PRESELECT_BATCH = 256

MAX_EPOCHS = 30
PATIENCE = 6

LR = 1e-3
WEIGHT_DECAY = 1e-4
TARGET_TEMP = 0.50

MAX_TRAIN_QUERIES_PER_PHASE = 60000

INVALID_SCORE = -1e9

BLOCK_LEN = 20
N_BOOT = 5000

FORCE_REBUILD_HORIZONS = False
FORCE_RETRAIN = False

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory GB:",
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )

print("Output:", RESULT_DIR)


## 1. Load base anchors and raw price data

In [ ]:

base = pd.read_parquet(
    BASE_WINDOW_FILE
)

base["EndDate"] = pd.to_datetime(
    base["EndDate"]
)

base["FutureEndDate"] = pd.to_datetime(
    base["FutureEndDate"]
)

raw = pd.read_parquet(
    RAW_FILE
)

raw["Date"] = pd.to_datetime(
    raw["Date"]
)

print(
    "Base anchors:",
    len(base),
    "| Tickers:",
    base["Ticker"].nunique(),
)

print(
    "Raw rows:",
    len(raw),
    "| Tickers:",
    raw["Ticker"].nunique(),
)

print(
    "Base range:",
    base["EndDate"].min(),
    "->",
    base["EndDate"].max(),
)

print(
    "Raw range:",
    raw["Date"].min(),
    "->",
    raw["Date"].max(),
)


## 2. Check required columns

In [ ]:

LOCAL_COLS = [
    "STK_RET_5",
    "STK_RET_20",
    "STK_RV_20",
    "STK_VOL_RATIO",
    "STK_DD_60",
    "REL_STRENGTH_20",
]

required_base = [
    "Ticker",
    "EndDate",
    "Pattern",
    "FuturePath",
    "FutureEndDate",
    "Sector",
] + LOCAL_COLS

missing_base = [
    c
    for c
    in required_base
    if c not in base.columns
]

assert not missing_base, (
    "Missing base-window columns: "
    f"{missing_base}"
)

required_raw = [
    "Ticker",
    "Date",
    "Close",
]

missing_raw = [
    c
    for c
    in required_raw
    if c not in raw.columns
]

assert not missing_raw, (
    "Missing raw-data columns: "
    f"{missing_raw}"
)

print("Required columns: OK")



## 3. Build horizon-specific future paths

For each ticker and each existing anchor `EndDate`:

1. locate the anchor close
2. take the next \(H\) trading-day closes
3. compute cumulative log return:

\[
\log P_{t+j} - \log P_t
\]

This is exactly the cumulative log-return path.

A validation check compares the reconstructed \(H=5\) target
against the original stored `FuturePath`.


In [ ]:

def build_horizon_windows(
    base_df,
    raw_df,
    horizon,
):
    feature_cols = [
        c
        for c
        in base_df.columns
        if c not in [
            "FuturePath",
            "FutureEndDate",
        ]
    ]

    anchor = (
        base_df[
            feature_cols
        ]
        .copy()
    )

    rows = []

    raw_groups = {
        str(t):
        g.sort_values(
            "Date"
        ).reset_index(
            drop=True
        )
        for t, g
        in raw_df.groupby(
            "Ticker"
        )
    }

    base_groups = (
        anchor.groupby(
            "Ticker"
        )
    )

    for ticker, g_anchor in tqdm(
        base_groups,
        total=anchor["Ticker"].nunique(),
        desc=f"Build H={horizon}",
    ):
        ticker = str(
            ticker
        )

        if ticker not in raw_groups:
            continue

        g_raw = raw_groups[
            ticker
        ]

        dates = (
            g_raw[
                "Date"
            ].to_numpy()
        )

        close = (
            g_raw[
                "Close"
            ].to_numpy(
                dtype=np.float64
            )
        )

        # Date -> exact integer location.
        date_to_pos = {
            pd.Timestamp(d):
            i
            for i, d
            in enumerate(
                dates
            )
        }

        for _, r in g_anchor.iterrows():

            end_date = pd.Timestamp(
                r[
                    "EndDate"
                ]
            )

            pos = date_to_pos.get(
                end_date
            )

            if pos is None:
                continue

            future_end_pos = (
                pos +
                horizon
            )

            if future_end_pos >= len(
                close
            ):
                continue

            p0 = close[
                pos
            ]

            future_close = close[
                pos + 1:
                pos + 1 + horizon
            ]

            if (
                not np.isfinite(
                    p0
                )
                or
                p0 <= 0
                or
                len(
                    future_close
                ) != horizon
                or
                not np.isfinite(
                    future_close
                ).all()
                or
                (
                    future_close <= 0
                ).any()
            ):
                continue

            future_path = (
                np.log(
                    future_close
                ) -
                np.log(
                    p0
                )
            ).astype(
                np.float32
            )

            out = r.to_dict()

            out[
                "FuturePath"
            ] = future_path

            out[
                "FutureEndDate"
            ] = pd.Timestamp(
                dates[
                    future_end_pos
                ]
            )

            out[
                "Horizon"
            ] = int(
                horizon
            )

            rows.append(
                out
            )

    out_df = pd.DataFrame(
        rows
    )

    out_df[
        "EndDate"
    ] = pd.to_datetime(
        out_df[
            "EndDate"
        ]
    )

    out_df[
        "FutureEndDate"
    ] = pd.to_datetime(
        out_df[
            "FutureEndDate"
        ]
    )

    return (
        out_df
        .sort_values(
            [
                "Ticker",
                "EndDate",
            ]
        )
        .reset_index(
            drop=True
        )
    )


HORIZON_WINDOWS = {}

for H in HORIZONS:

    cache_path = (
        CACHE_DIR /
        f"windows_H{H}.parquet"
    )

    if (
        cache_path.exists()
        and not FORCE_REBUILD_HORIZONS
    ):

        df_h = pd.read_parquet(
            cache_path
        )

        df_h[
            "EndDate"
        ] = pd.to_datetime(
            df_h[
                "EndDate"
            ]
        )

        df_h[
            "FutureEndDate"
        ] = pd.to_datetime(
            df_h[
                "FutureEndDate"
            ]
        )

        print(
            f"Loaded cached H={H}:",
            len(
                df_h
            )
        )

    else:

        df_h = (
            build_horizon_windows(
                base,
                raw,
                H,
            )
        )

        df_h.to_parquet(
            cache_path,
            index=False,
        )

        print(
            f"Saved H={H}:",
            len(
                df_h
            )
        )

    HORIZON_WINDOWS[
        H
    ] = df_h


## 4. Validate reconstructed H=5 target against the original windows

In [ ]:

original_h5 = (
    base[
        [
            "Ticker",
            "EndDate",
            "FuturePath",
        ]
    ]
    .rename(
        columns={
            "FuturePath":
                "FuturePath_Original"
        }
    )
)

recon_h5 = (
    HORIZON_WINDOWS[
        5
    ][
        [
            "Ticker",
            "EndDate",
            "FuturePath",
        ]
    ]
    .rename(
        columns={
            "FuturePath":
                "FuturePath_Reconstructed"
        }
    )
)

check_h5 = original_h5.merge(
    recon_h5,
    on=[
        "Ticker",
        "EndDate",
    ],
    how="inner",
)

def max_abs_path_diff(row):
    a = np.asarray(
        row[
            "FuturePath_Original"
        ],
        dtype=np.float64,
    )

    b = np.asarray(
        row[
            "FuturePath_Reconstructed"
        ],
        dtype=np.float64,
    )

    return float(
        np.max(
            np.abs(
                a -
                b
            )
        )
    )

sample_check = (
    check_h5.sample(
        n=min(
            5000,
            len(
                check_h5
            ),
        ),
        random_state=42,
    )
    .copy()
)

sample_check[
    "MaxAbsDiff"
] = (
    sample_check.apply(
        max_abs_path_diff,
        axis=1,
    )
)

print(
    "H=5 reconstruction sample size:",
    len(
        sample_check
    )
)

print(
    "Median max abs diff:",
    sample_check[
        "MaxAbsDiff"
    ].median()
)

print(
    "99th percentile:",
    sample_check[
        "MaxAbsDiff"
    ].quantile(
        0.99
    )
)

print(
    "Max:",
    sample_check[
        "MaxAbsDiff"
    ].max()
)

assert (
    sample_check[
        "MaxAbsDiff"
    ].quantile(
        0.99
    ) < 1e-4
), (
    "H=5 reconstructed target does not match "
    "the original pipeline closely enough. "
    "Inspect raw Close convention before continuing."
)


## 5. Horizon coverage summary

In [ ]:

coverage_rows = []

for H, df_h in (
    HORIZON_WINDOWS.items()
):

    coverage_rows.append({
        "Horizon":
            H,

        "Windows":
            len(
                df_h
            ),

        "Tickers":
            df_h[
                "Ticker"
            ].nunique(),

        "FirstEndDate":
            df_h[
                "EndDate"
            ].min(),

        "LastEndDate":
            df_h[
                "EndDate"
            ].max(),

        "LastFutureEndDate":
            df_h[
                "FutureEndDate"
            ].max(),
    })

coverage_table = pd.DataFrame(
    coverage_rows
)

display(
    coverage_table
)

coverage_table.to_csv(
    RESULT_DIR /
    "01_horizon_coverage.csv",
    index=False,
)


## 6. Temporal split helper

In [ ]:

TRAIN_MEMORY_CUTOFF = pd.Timestamp(
    "2009-12-31"
)

TRAIN_START = pd.Timestamp(
    "2010-01-01"
)

TRAIN_END = pd.Timestamp(
    "2014-12-31"
)

VAL_MEMORY_CUTOFF = pd.Timestamp(
    "2014-12-31"
)

VAL_START = pd.Timestamp(
    "2015-01-01"
)

VAL_END = pd.Timestamp(
    "2019-12-31"
)

TEST_MEMORY_CUTOFF = pd.Timestamp(
    "2019-12-31"
)

TEST_START = pd.Timestamp(
    "2020-01-01"
)

def make_splits(
    df,
):
    train_cand = df[
        df[
            "FutureEndDate"
        ] <=
        TRAIN_MEMORY_CUTOFF
    ].reset_index(
        drop=True
    )

    train_query = df[
        (
            df[
                "EndDate"
            ] >=
            TRAIN_START
        ) &
        (
            df[
                "FutureEndDate"
            ] <=
            TRAIN_END
        )
    ].reset_index(
        drop=True
    )

    val_cand = df[
        df[
            "FutureEndDate"
        ] <=
        VAL_MEMORY_CUTOFF
    ].reset_index(
        drop=True
    )

    val_query = df[
        (
            df[
                "EndDate"
            ] >=
            VAL_START
        ) &
        (
            df[
                "FutureEndDate"
            ] <=
            VAL_END
        )
    ].reset_index(
        drop=True
    )

    test_cand = df[
        df[
            "FutureEndDate"
        ] <=
        TEST_MEMORY_CUTOFF
    ].reset_index(
        drop=True
    )

    test_query = df[
        df[
            "EndDate"
        ] >=
        TEST_START
    ].reset_index(
        drop=True
    )

    return {
        "train_cand":
            train_cand,

        "train_query":
            train_query,

        "val_cand":
            val_cand,

        "val_query":
            val_query,

        "test_cand":
            test_cand,

        "test_query":
            test_query,
    }

SPLITS = {
    H:
    make_splits(
        HORIZON_WINDOWS[
            H
        ]
    )
    for H in HORIZONS
}


## 7. Split summary by horizon

In [ ]:

split_rows = []

for H in HORIZONS:

    sp = SPLITS[
        H
    ]

    for name in [
        "train_cand",
        "train_query",
        "val_cand",
        "val_query",
        "test_cand",
        "test_query",
    ]:

        d = sp[
            name
        ]

        split_rows.append({
            "Horizon":
                H,

            "Split":
                name,

            "N":
                len(
                    d
                ),

            "Tickers":
                d[
                    "Ticker"
                ].nunique(),
        })

split_summary = pd.DataFrame(
    split_rows
)

display(
    split_summary
)

split_summary.to_csv(
    RESULT_DIR /
    "02_split_summary.csv",
    index=False,
)


## 8. Tensor helpers

In [ ]:

def stack_col(
    df,
    col,
):
    return np.stack(
        df[
            col
        ].to_numpy()
    ).astype(
        np.float32
    )

def basic_tensors(
    cand_df,
    query_df,
):
    return {
        "C_PATTERN":
            torch.tensor(
                stack_col(
                    cand_df,
                    "Pattern",
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "Q_PATTERN":
            torch.tensor(
                stack_col(
                    query_df,
                    "Pattern",
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "C_FUTURE":
            torch.tensor(
                stack_col(
                    cand_df,
                    "FuturePath",
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "Q_FUTURE":
            torch.tensor(
                stack_col(
                    query_df,
                    "FuturePath",
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),
    }

BASIC = {}

for H in HORIZONS:

    sp = SPLITS[
        H
    ]

    BASIC[
        H
    ] = {
        "train":
            basic_tensors(
                sp[
                    "train_cand"
                ],
                sp[
                    "train_query"
                ],
            ),

        "val":
            basic_tensors(
                sp[
                    "val_cand"
                ],
                sp[
                    "val_query"
                ],
            ),

        "test":
            basic_tensors(
                sp[
                    "test_cand"
                ],
                sp[
                    "test_query"
                ],
            ),
    }


## 9. Local-only robust context scaling

In [ ]:

def fit_robust_scaler(
    df,
    cols,
):
    x = df[
        cols
    ].to_numpy(
        dtype=np.float32
    )

    med = np.nanmedian(
        x,
        axis=0,
    )

    q25 = np.nanpercentile(
        x,
        25,
        axis=0,
    )

    q75 = np.nanpercentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75 -
        q25
    )

    iqr = np.where(
        iqr < 1e-6,
        1.0,
        iqr,
    )

    return (
        med.astype(
            np.float32
        ),
        iqr.astype(
            np.float32
        ),
    )

def transform_context(
    df,
    cols,
    med,
    iqr,
    clip=8.0,
):
    x = df[
        cols
    ].to_numpy(
        dtype=np.float32
    )

    x = (
        x -
        med
    ) / iqr

    x = np.clip(
        x,
        -clip,
        clip,
    )

    return x.astype(
        np.float32
    )

CONTEXT = {}

for H in HORIZONS:

    sp = SPLITS[
        H
    ]

    med, iqr = (
        fit_robust_scaler(
            sp[
                "train_cand"
            ],
            LOCAL_COLS,
        )
    )

    CONTEXT[
        H
    ] = {
        "median":
            med,

        "iqr":
            iqr,

        "train_c":
            torch.tensor(
                transform_context(
                    sp[
                        "train_cand"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "train_q":
            torch.tensor(
                transform_context(
                    sp[
                        "train_query"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "val_c":
            torch.tensor(
                transform_context(
                    sp[
                        "val_cand"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "val_q":
            torch.tensor(
                transform_context(
                    sp[
                        "val_query"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "test_c":
            torch.tensor(
                transform_context(
                    sp[
                        "test_cand"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),

        "test_q":
            torch.tensor(
                transform_context(
                    sp[
                        "test_query"
                    ],
                    LOCAL_COLS,
                    med,
                    iqr,
                ),
                dtype=torch.float32,
                device=DEVICE,
            ),
    }


## 10. CrossStock Top-100 pattern preselection

In [ ]:

@torch.no_grad()
def cross_preselect(
    q_pattern,
    c_pattern,
    top_m=100,
    batch_size=256,
    desc="Cross preselect",
):
    idx_list = []
    score_list = []

    for start in tqdm(
        range(
            0,
            len(
                q_pattern
            ),
            batch_size,
        ),
        desc=desc,
    ):

        end = min(
            start +
            batch_size,
            len(
                q_pattern
            ),
        )

        score = (
            q_pattern[
                start:end
            ] @
            c_pattern.T
        )

        top_score, top_idx = (
            torch.topk(
                score,
                k=min(
                    top_m,
                    c_pattern.shape[
                        0
                    ],
                ),
                dim=1,
                largest=True,
            )
        )

        idx_list.append(
            top_idx.cpu()
        )

        score_list.append(
            top_score.cpu()
        )

        del score
        del top_score
        del top_idx

    return (
        torch.cat(
            idx_list,
            dim=0,
        ),
        torch.cat(
            score_list,
            dim=0,
        ),
    )

CROSS_PRE = {}

for H in HORIZONS:

    cache_path = (
        CACHE_DIR /
        f"cross_preselect_H{H}_M{TOP_M}.pt"
    )

    if cache_path.exists():

        c = torch.load(
            cache_path,
            map_location="cpu",
            weights_only=False,
        )

        CROSS_PRE[
            H
        ] = c

        print(
            f"Loaded Cross H={H}"
        )

        continue

    hdict = {}

    for phase in [
        "train",
        "val",
        "test",
    ]:

        b = BASIC[
            H
        ][
            phase
        ]

        idx, score = (
            cross_preselect(
                b[
                    "Q_PATTERN"
                ],
                b[
                    "C_PATTERN"
                ],
                top_m=TOP_M,
                batch_size=PRESELECT_BATCH,
                desc=(
                    f"H={H} "
                    f"{phase} Cross Top-{TOP_M}"
                ),
            )
        )

        hdict[
            f"{phase}_idx"
        ] = idx

        hdict[
            f"{phase}_score"
        ] = score

    torch.save(
        hdict,
        cache_path,
    )

    CROSS_PRE[
        H
    ] = hdict

    print(
        "Saved:",
        cache_path
    )


## 11. SameStock Top-100 pattern preselection

In [ ]:

def same_preselect(
    cand_df,
    query_df,
    c_pattern,
    q_pattern,
    top_m=100,
):
    cand_groups = {
        str(t):
            np.asarray(
                idx,
                dtype=np.int64,
            )
        for t, idx
        in cand_df.groupby(
            "Ticker"
        ).indices.items()
    }

    query_groups = (
        query_df.groupby(
            "Ticker"
        ).indices
    )

    idx_all = np.full(
        (
            len(
                query_df
            ),
            top_m,
        ),
        -1,
        dtype=np.int64,
    )

    score_all = np.full(
        (
            len(
                query_df
            ),
            top_m,
        ),
        INVALID_SCORE,
        dtype=np.float32,
    )

    counts = np.zeros(
        len(
            query_df
        ),
        dtype=np.int32,
    )

    for ticker, qids in tqdm(
        query_groups.items(),
        total=len(
            query_groups
        ),
        desc="SameStock preselect",
    ):

        ticker = str(
            ticker
        )

        qids = np.asarray(
            qids,
            dtype=np.int64,
        )

        cids = cand_groups.get(
            ticker
        )

        if cids is None:
            continue

        k = min(
            top_m,
            len(
                cids
            ),
        )

        score = (
            q_pattern[
                qids
            ] @
            c_pattern[
                cids
            ].T
        )

        top_score, local = (
            torch.topk(
                score,
                k=k,
                dim=1,
                largest=True,
            )
        )

        cids_t = torch.tensor(
            cids,
            dtype=torch.long,
            device=DEVICE,
        )

        global_idx = (
            cids_t[
                local
            ]
        )

        idx_all[
            qids,
            :k
        ] = (
            global_idx
            .cpu()
            .numpy()
        )

        score_all[
            qids,
            :k
        ] = (
            top_score
            .cpu()
            .numpy()
        )

        counts[
            qids
        ] = len(
            cids
        )

    return {
        "idx":
            torch.tensor(
                idx_all,
                dtype=torch.long,
            ),

        "score":
            torch.tensor(
                score_all,
                dtype=torch.float32,
            ),

        "count":
            counts,
    }

SAME_PRE = {}

for H in HORIZONS:

    cache_path = (
        CACHE_DIR /
        f"same_preselect_H{H}_M{TOP_M}.pt"
    )

    if cache_path.exists():

        SAME_PRE[
            H
        ] = torch.load(
            cache_path,
            map_location="cpu",
            weights_only=False,
        )

        print(
            f"Loaded Same H={H}"
        )

        continue

    sp = SPLITS[
        H
    ]

    hdict = {}

    for phase in [
        "train",
        "val",
        "test",
    ]:

        cand_df = sp[
            f"{phase}_cand"
        ]

        query_df = sp[
            f"{phase}_query"
        ]

        b = BASIC[
            H
        ][
            phase
        ]

        r = same_preselect(
            cand_df,
            query_df,
            b[
                "C_PATTERN"
            ],
            b[
                "Q_PATTERN"
            ],
            top_m=TOP_M,
        )

        hdict[
            f"{phase}_idx"
        ] = r[
            "idx"
        ]

        hdict[
            f"{phase}_score"
        ] = r[
            "score"
        ]

        hdict[
            f"{phase}_count"
        ] = r[
            "count"
        ]

    torch.save(
        hdict,
        cache_path,
    )

    SAME_PRE[
        H
    ] = hdict

    print(
        "Saved:",
        cache_path
    )


## 12. SameStock coverage

In [ ]:

same_coverage_rows = []

for H in HORIZONS:

    for phase in [
        "train",
        "val",
        "test",
    ]:

        count = SAME_PRE[
            H
        ][
            f"{phase}_count"
        ]

        eligible = (
            count >=
            TOP_K
        )

        same_coverage_rows.append({
            "Horizon":
                H,

            "Split":
                phase,

            "Queries":
                len(
                    count
                ),

            "Eligible":
                int(
                    eligible.sum()
                ),

            "Coverage":
                float(
                    eligible.mean()
                ),
        })

same_coverage = pd.DataFrame(
    same_coverage_rows
)

display(
    same_coverage
)

same_coverage.to_csv(
    RESULT_DIR /
    "03_same_stock_coverage.csv",
    index=False,
)


## 13. Learned reranker

In [ ]:

class FutureCompatibilityReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim // 2,
                1,
            ),
        )

        raw = math.log(
            math.exp(
                initial_alpha
            ) -
            1.0
        )

        self.log_alpha_raw = (
            nn.Parameter(
                torch.tensor(
                    raw,
                    dtype=torch.float32,
                )
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.log_alpha_raw
        )

    def forward(
        self,
        pattern_score,
        query_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            query_context[
                :, None, :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ..., None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(-1)
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )


## 14. Stable listwise loss

In [ ]:

def stable_listwise_loss(
    score_finite,
    future_dist_finite,
    valid=None,
):
    assert torch.isfinite(
        score_finite
    ).all()

    assert torch.isfinite(
        future_dist_finite
    ).all()

    if valid is None:

        mean = future_dist_finite.mean(
            dim=1,
            keepdim=True,
        )

        std = future_dist_finite.std(
            dim=1,
            keepdim=True,
            unbiased=False,
        ).clamp_min(
            1e-6
        )

        z = (
            future_dist_finite -
            mean
        ) / std

        target = torch.softmax(
            -z /
            TARGET_TEMP,
            dim=1,
        )

        log_prob = F.log_softmax(
            score_finite,
            dim=1,
        )

        loss = -(
            target *
            log_prob
        ).sum(
            dim=1
        ).mean()

        assert torch.isfinite(
            loss
        )

        return loss

    valid_f = valid.float()

    n_valid = (
        valid_f.sum(
            dim=1,
            keepdim=True,
        ).clamp_min(
            1.0
        )
    )

    mean = (
        (
            future_dist_finite *
            valid_f
        ).sum(
            dim=1,
            keepdim=True,
        ) /
        n_valid
    )

    centered = (
        future_dist_finite -
        mean
    )

    var = (
        (
            centered.pow(
                2
            ) *
            valid_f
        ).sum(
            dim=1,
            keepdim=True,
        ) /
        n_valid
    )

    std = (
        torch.sqrt(
            var
        ).clamp_min(
            1e-6
        )
    )

    z = (
        future_dist_finite -
        mean
    ) / std

    target_logits = torch.where(
        valid,
        -z /
        TARGET_TEMP,
        torch.full_like(
            z,
            INVALID_SCORE,
        ),
    )

    model_logits = torch.where(
        valid,
        score_finite,
        torch.full_like(
            score_finite,
            INVALID_SCORE,
        ),
    )

    target = torch.softmax(
        target_logits,
        dim=1,
    )

    log_prob = F.log_softmax(
        model_logits,
        dim=1,
    )

    target_safe = torch.where(
        valid,
        target,
        torch.zeros_like(
            target
        ),
    )

    log_prob_safe = torch.where(
        valid,
        log_prob,
        torch.zeros_like(
            log_prob
        ),
    )

    loss = -(
        target_safe *
        log_prob_safe
    ).sum(
        dim=1
    ).mean()

    assert torch.isfinite(
        loss
    )

    return loss


## 15. Batch helpers

In [ ]:

def prepare_cross_batch(
    qids_cpu,
    pre_idx,
    pre_score,
    q_context,
    c_context,
    q_future,
    c_future,
):
    idx = (
        pre_idx[
            qids_cpu
        ].to(
            DEVICE
        )
    )

    ps = (
        pre_score[
            qids_cpu
        ].to(
            DEVICE
        )
    )

    qids = torch.tensor(
        qids_cpu,
        dtype=torch.long,
        device=DEVICE,
    )

    qc = q_context[
        qids
    ]

    cc = c_context[
        idx
    ]

    qf = q_future[
        qids
    ]

    cf = c_future[
        idx
    ]

    fd = (
        (
            cf -
            qf[
                :, None, :
            ]
        ) ** 2
    ).mean(
        dim=2
    )

    return (
        idx,
        ps,
        qc,
        cc,
        fd,
        None,
    )


def prepare_same_batch(
    qids_cpu,
    pre_idx,
    pre_score,
    q_context,
    c_context,
    q_future,
    c_future,
):
    idx_cpu = pre_idx[
        qids_cpu
    ]

    valid_cpu = (
        idx_cpu >=
        0
    )

    safe_idx = (
        idx_cpu
        .clamp_min(
            0
        )
        .to(
            DEVICE
        )
    )

    valid = (
        valid_cpu.to(
            DEVICE
        )
    )

    raw_score = (
        pre_score[
            qids_cpu
        ].to(
            DEVICE
        )
    )

    ps = torch.where(
        valid,
        raw_score,
        torch.zeros_like(
            raw_score
        ),
    )

    qids = torch.tensor(
        qids_cpu,
        dtype=torch.long,
        device=DEVICE,
    )

    qc = q_context[
        qids
    ]

    cc = c_context[
        safe_idx
    ]

    cc = torch.where(
        valid[
            ..., None
        ],
        cc,
        torch.zeros_like(
            cc
        ),
    )

    qf = q_future[
        qids
    ]

    cf = c_future[
        safe_idx
    ]

    fd = (
        (
            cf -
            qf[
                :, None, :
            ]
        ) ** 2
    ).mean(
        dim=2
    )

    fd = torch.where(
        valid,
        fd,
        torch.zeros_like(
            fd
        ),
    )

    return (
        safe_idx,
        ps,
        qc,
        cc,
        fd,
        valid,
    )


## 16. Metric helpers

In [ ]:

@torch.no_grad()
def metrics_from_indices(
    idx_cpu,
    q_future,
    c_future,
):
    idx = idx_cpu.to(
        DEVICE
    )

    retrieved = c_future[
        idx
    ]

    true = q_future

    analog_mse = (
        (
            retrieved -
            true[
                :, None, :
            ]
        ) ** 2
    ).mean(
        dim=(1, 2)
    )

    pred = retrieved.mean(
        dim=1
    )

    forecast_mse = (
        (
            pred -
            true
        ) ** 2
    ).mean(
        dim=1
    )

    terminal_mae = (
        pred[
            :, -1
        ] -
        true[
            :, -1
        ]
    ).abs()

    direction = (
        torch.sign(
            pred[
                :, -1
            ]
        ) ==
        torch.sign(
            true[
                :, -1
            ]
        )
    ).float()

    return pd.DataFrame({
        "AnalogFutureMSE":
            analog_mse
            .cpu()
            .numpy(),

        "ForecastMSE":
            forecast_mse
            .cpu()
            .numpy(),

        "TerminalMAE":
            terminal_mae
            .cpu()
            .numpy(),

        "DirectionCorrect":
            direction
            .cpu()
            .numpy(),
    })


def summarize_metrics(
    df,
):
    return {
        "AnalogFutureMSE":
            float(
                df[
                    "AnalogFutureMSE"
                ].mean()
            ),

        "ForecastMSE":
            float(
                df[
                    "ForecastMSE"
                ].mean()
            ),

        "TerminalMAE":
            float(
                df[
                    "TerminalMAE"
                ].mean()
            ),

        "DirectionAcc":
            float(
                df[
                    "DirectionCorrect"
                ].mean()
            ),
    }


## 17. Evaluation function

In [ ]:

@torch.no_grad()
def evaluate_learned(
    model,
    mode,
    qids_all,
    pre_idx,
    pre_score,
    q_context,
    c_context,
    q_future,
    c_future,
):
    model.eval()

    selected_all = []

    for start in range(
        0,
        len(
            qids_all
        ),
        EVAL_BATCH,
    ):

        qids = qids_all[
            start:
            start +
            EVAL_BATCH
        ]

        prepare_fn = (
            prepare_cross_batch
            if mode ==
            "cross"
            else
            prepare_same_batch
        )

        (
            idx,
            ps,
            qc,
            cc,
            _,
            valid,
        ) = prepare_fn(
            qids,
            pre_idx,
            pre_score,
            q_context,
            c_context,
            q_future,
            c_future,
        )

        score = model(
            ps,
            qc,
            cc,
        )

        if valid is not None:

            score = torch.where(
                valid,
                score,
                torch.full_like(
                    score,
                    INVALID_SCORE,
                ),
            )

        local = torch.topk(
            score,
            k=TOP_K,
            dim=1,
            largest=True,
        ).indices

        selected = torch.gather(
            idx,
            1,
            local,
        )

        selected_all.append(
            selected.cpu()
        )

    selected_idx = torch.cat(
        selected_all,
        dim=0,
    )

    qids_t = torch.tensor(
        qids_all,
        dtype=torch.long,
        device=DEVICE,
    )

    metrics = metrics_from_indices(
        selected_idx,
        q_future[
            qids_t
        ],
        c_future,
    )

    return (
        selected_idx,
        metrics,
    )


## 18. Training helpers

In [ ]:

def set_seed(
    seed
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


def sample_qids(
    eligible_qids,
    max_n,
    seed,
):
    qids = np.asarray(
        eligible_qids,
        dtype=np.int64,
    )

    if len(
        qids
    ) <= max_n:
        return qids

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            qids,
            size=max_n,
            replace=False,
        )
    )


def assert_model_finite(
    model,
    where="",
):
    assert torch.isfinite(
        model.alpha
    ), (
        f"Non-finite alpha at {where}"
    )

    for name, p in (
        model.named_parameters()
    ):

        assert torch.isfinite(
            p
        ).all(), (
            f"Non-finite parameter "
            f"{name} at {where}"
        )


## 19. Unified phase-A training

In [ ]:

def train_phase_a(
    H,
    mode,
    seed,
):
    set_seed(
        seed
    )

    model = (
        FutureCompatibilityReranker(
            context_dim=len(
                LOCAL_COLS
            )
        )
        .to(
            DEVICE
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    if mode == "cross":

        train_idx = CROSS_PRE[
            H
        ][
            "train_idx"
        ]

        train_score = CROSS_PRE[
            H
        ][
            "train_score"
        ]

        val_idx = CROSS_PRE[
            H
        ][
            "val_idx"
        ]

        val_score = CROSS_PRE[
            H
        ][
            "val_score"
        ]

        train_eligible = np.arange(
            len(
                train_idx
            ),
            dtype=np.int64,
        )

        val_eligible = np.arange(
            len(
                val_idx
            ),
            dtype=np.int64,
        )

        prepare_fn = (
            prepare_cross_batch
        )

    else:

        train_idx = SAME_PRE[
            H
        ][
            "train_idx"
        ]

        train_score = SAME_PRE[
            H
        ][
            "train_score"
        ]

        val_idx = SAME_PRE[
            H
        ][
            "val_idx"
        ]

        val_score = SAME_PRE[
            H
        ][
            "val_score"
        ]

        train_eligible = np.where(
            SAME_PRE[
                H
            ][
                "train_count"
            ] >=
            TOP_K
        )[0]

        val_eligible = np.where(
            SAME_PRE[
                H
            ][
                "val_count"
            ] >=
            TOP_K
        )[0]

        prepare_fn = (
            prepare_same_batch
        )

    train_qids = sample_qids(
        train_eligible,
        MAX_TRAIN_QUERIES_PER_PHASE,
        seed=(
            10000 +
            H * 100 +
            seed
        ),
    )

    ctx = CONTEXT[
        H
    ]

    bt = BASIC[
        H
    ]

    best_epoch = None
    best_val_mse = float(
        "inf"
    )

    wait = 0
    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        model.train()

        perm = np.random.permutation(
            train_qids
        )

        losses = []

        for start in range(
            0,
            len(
                perm
            ),
            TRAIN_BATCH,
        ):

            qids = perm[
                start:
                start +
                TRAIN_BATCH
            ]

            (
                _,
                ps,
                qc,
                cc,
                fd,
                valid,
            ) = prepare_fn(
                qids,
                train_idx,
                train_score,
                ctx[
                    "train_q"
                ],
                ctx[
                    "train_c"
                ],
                bt[
                    "train"
                ][
                    "Q_FUTURE"
                ],
                bt[
                    "train"
                ][
                    "C_FUTURE"
                ],
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            score = model(
                ps,
                qc,
                cc,
            )

            loss = stable_listwise_loss(
                score,
                fd,
                valid=valid,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            assert_model_finite(
                model,
                where=(
                    f"H={H} "
                    f"{mode} "
                    f"seed={seed} "
                    f"epoch={epoch}"
                ),
            )

            losses.append(
                float(
                    loss.item()
                )
            )

        _, val_m = evaluate_learned(
            model,
            mode,
            val_eligible,
            val_idx,
            val_score,
            ctx[
                "val_q"
            ],
            ctx[
                "val_c"
            ],
            bt[
                "val"
            ][
                "Q_FUTURE"
            ],
            bt[
                "val"
            ][
                "C_FUTURE"
            ],
        )

        val_s = summarize_metrics(
            val_m
        )

        val_mse = val_s[
            "ForecastMSE"
        ]

        history.append({
            "Horizon":
                H,

            "Mode":
                mode,

            "Seed":
                seed,

            "Epoch":
                epoch,

            "TrainN":
                len(
                    train_qids
                ),

            "TrainLoss":
                float(
                    np.mean(
                        losses
                    )
                ),

            "Alpha":
                float(
                    model.alpha.item()
                ),

            "ValAnalogFutureMSE":
                val_s[
                    "AnalogFutureMSE"
                ],

            "ValForecastMSE":
                val_s[
                    "ForecastMSE"
                ],

            "ValTerminalMAE":
                val_s[
                    "TerminalMAE"
                ],

            "ValDirectionAcc":
                val_s[
                    "DirectionAcc"
                ],
        })

        if (
            val_mse <
            best_val_mse -
            1e-9
        ):

            best_val_mse = (
                val_mse
            )

            best_epoch = (
                epoch
            )

            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    return (
        best_epoch,
        best_val_mse,
        pd.DataFrame(
            history
        ),
    )


## 20. Unified refit

In [ ]:

def refit_model(
    H,
    mode,
    seed,
    epochs,
):
    set_seed(
        seed
    )

    model = (
        FutureCompatibilityReranker(
            context_dim=len(
                LOCAL_COLS
            )
        )
        .to(
            DEVICE
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    ctx = CONTEXT[
        H
    ]

    bt = BASIC[
        H
    ]

    if mode == "cross":

        train_eligible = np.arange(
            len(
                CROSS_PRE[
                    H
                ][
                    "train_idx"
                ]
            ),
            dtype=np.int64,
        )

        val_eligible = np.arange(
            len(
                CROSS_PRE[
                    H
                ][
                    "val_idx"
                ]
            ),
            dtype=np.int64,
        )

        phase_specs = [
            {
                "idx":
                    CROSS_PRE[
                        H
                    ][
                        "train_idx"
                    ],

                "score":
                    CROSS_PRE[
                        H
                    ][
                        "train_score"
                    ],

                "qctx":
                    ctx[
                        "train_q"
                    ],

                "cctx":
                    ctx[
                        "train_c"
                    ],

                "qf":
                    bt[
                        "train"
                    ][
                        "Q_FUTURE"
                    ],

                "cf":
                    bt[
                        "train"
                    ][
                        "C_FUTURE"
                    ],

                "eligible":
                    train_eligible,
            },

            {
                "idx":
                    CROSS_PRE[
                        H
                    ][
                        "val_idx"
                    ],

                "score":
                    CROSS_PRE[
                        H
                    ][
                        "val_score"
                    ],

                "qctx":
                    ctx[
                        "val_q"
                    ],

                "cctx":
                    ctx[
                        "val_c"
                    ],

                "qf":
                    bt[
                        "val"
                    ][
                        "Q_FUTURE"
                    ],

                "cf":
                    bt[
                        "val"
                    ][
                        "C_FUTURE"
                    ],

                "eligible":
                    val_eligible,
            },
        ]

        prepare_fn = (
            prepare_cross_batch
        )

    else:

        train_eligible = np.where(
            SAME_PRE[
                H
            ][
                "train_count"
            ] >=
            TOP_K
        )[0]

        val_eligible = np.where(
            SAME_PRE[
                H
            ][
                "val_count"
            ] >=
            TOP_K
        )[0]

        phase_specs = [
            {
                "idx":
                    SAME_PRE[
                        H
                    ][
                        "train_idx"
                    ],

                "score":
                    SAME_PRE[
                        H
                    ][
                        "train_score"
                    ],

                "qctx":
                    ctx[
                        "train_q"
                    ],

                "cctx":
                    ctx[
                        "train_c"
                    ],

                "qf":
                    bt[
                        "train"
                    ][
                        "Q_FUTURE"
                    ],

                "cf":
                    bt[
                        "train"
                    ][
                        "C_FUTURE"
                    ],

                "eligible":
                    train_eligible,
            },

            {
                "idx":
                    SAME_PRE[
                        H
                    ][
                        "val_idx"
                    ],

                "score":
                    SAME_PRE[
                        H
                    ][
                        "val_score"
                    ],

                "qctx":
                    ctx[
                        "val_q"
                    ],

                "cctx":
                    ctx[
                        "val_c"
                    ],

                "qf":
                    bt[
                        "val"
                    ][
                        "Q_FUTURE"
                    ],

                "cf":
                    bt[
                        "val"
                    ][
                        "C_FUTURE"
                    ],

                "eligible":
                    val_eligible,
            },
        ]

        prepare_fn = (
            prepare_same_batch
        )

    for epoch in range(
        1,
        epochs + 1,
    ):

        order = [
            0,
            1,
        ]

        random.Random(
            H *
            10000 +
            seed *
            100 +
            epoch
        ).shuffle(
            order
        )

        model.train()

        for phase_id in order:

            p = phase_specs[
                phase_id
            ]

            qids = sample_qids(
                p[
                    "eligible"
                ],
                MAX_TRAIN_QUERIES_PER_PHASE,
                seed=(
                    H *
                    100000 +
                    seed *
                    1000 +
                    phase_id *
                    100 +
                    epoch
                ),
            )

            perm = np.random.permutation(
                qids
            )

            for start in range(
                0,
                len(
                    perm
                ),
                TRAIN_BATCH,
            ):

                qb = perm[
                    start:
                    start +
                    TRAIN_BATCH
                ]

                (
                    _,
                    ps,
                    qc,
                    cc,
                    fd,
                    valid,
                ) = prepare_fn(
                    qb,
                    p[
                        "idx"
                    ],
                    p[
                        "score"
                    ],
                    p[
                        "qctx"
                    ],
                    p[
                        "cctx"
                    ],
                    p[
                        "qf"
                    ],
                    p[
                        "cf"
                    ],
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                score = model(
                    ps,
                    qc,
                    cc,
                )

                loss = stable_listwise_loss(
                    score,
                    fd,
                    valid=valid,
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                optimizer.step()

                assert_model_finite(
                    model,
                    where=(
                        f"refit H={H} "
                        f"{mode} "
                        f"seed={seed}"
                    ),
                )

    return model


## 21. Train or load all SameStock and CrossStock learned models

The full run contains

\[
3\ \text{horizons}\times 2\ \text{memory modes}\times 3\ \text{seeds}=18\ \text{models}.
\]

Each training phase uses at most 60,000 queries. Existing checkpoints are reused automatically unless retraining is forced.


In [ ]:

LEARNED_RESULTS = {}
TRAIN_HISTORY = []

for H in HORIZONS:

    for mode in [
        "same",
        "cross",
    ]:

        for seed in SEEDS:

            ckpt_path = (
                MODEL_DIR /
                f"H{H}_{mode}_LocalOnly_M{TOP_M}_seed{seed}.pt"
            )

            print(
                "\n",
                "=" * 90
            )

            print(
                f"H={H} | {mode} | seed={seed}"
            )

            print(
                "=" * 90
            )

            if (
                ckpt_path.exists()
                and not FORCE_RETRAIN
            ):

                ckpt = torch.load(
                    ckpt_path,
                    map_location="cpu",
                    weights_only=False,
                )

                model = (
                    FutureCompatibilityReranker(
                        context_dim=len(
                            LOCAL_COLS
                        )
                    )
                    .to(
                        DEVICE
                    )
                )

                model.load_state_dict(
                    ckpt[
                        "StateDict"
                    ]
                )

                model.eval()

                best_epoch = int(
                    ckpt[
                        "BestEpoch"
                    ]
                )

                best_val = float(
                    ckpt[
                        "BestValidationMSE"
                    ]
                )

                print(
                    "Loaded checkpoint."
                )

            else:

                (
                    best_epoch,
                    best_val,
                    hist,
                ) = train_phase_a(
                    H,
                    mode,
                    seed,
                )

                TRAIN_HISTORY.append(
                    hist
                )

                print(
                    "Best epoch:",
                    best_epoch,
                    "| Val MSE:",
                    best_val,
                )

                model = refit_model(
                    H,
                    mode,
                    seed,
                    best_epoch,
                )

                torch.save(
                    {
                        "Horizon":
                            H,

                        "Mode":
                            mode,

                        "M":
                            TOP_M,

                        "K":
                            TOP_K,

                        "Seed":
                            seed,

                        "BestEpoch":
                            best_epoch,

                        "BestValidationMSE":
                            best_val,

                        "StateDict":
                            model.state_dict(),

                        "ContextCols":
                            LOCAL_COLS,

                        "Median":
                            CONTEXT[
                                H
                            ][
                                "median"
                            ],

                        "IQR":
                            CONTEXT[
                                H
                            ][
                                "iqr"
                            ],
                    },
                    ckpt_path,
                )

            assert_model_finite(
                model,
                where=(
                    f"final H={H} "
                    f"{mode} "
                    f"seed={seed}"
                ),
            )

            if mode == "cross":

                test_idx = CROSS_PRE[
                    H
                ][
                    "test_idx"
                ]

                test_score = CROSS_PRE[
                    H
                ][
                    "test_score"
                ]

                qids = np.arange(
                    len(
                        test_idx
                    ),
                    dtype=np.int64,
                )

            else:

                test_idx = SAME_PRE[
                    H
                ][
                    "test_idx"
                ]

                test_score = SAME_PRE[
                    H
                ][
                    "test_score"
                ]

                qids = np.where(
                    SAME_PRE[
                        H
                    ][
                        "test_count"
                    ] >=
                    TOP_K
                )[0]

            selected, metrics = (
                evaluate_learned(
                    model,
                    mode,
                    qids,
                    test_idx,
                    test_score,
                    CONTEXT[
                        H
                    ][
                        "test_q"
                    ],
                    CONTEXT[
                        H
                    ][
                        "test_c"
                    ],
                    BASIC[
                        H
                    ][
                        "test"
                    ][
                        "Q_FUTURE"
                    ],
                    BASIC[
                        H
                    ][
                        "test"
                    ][
                        "C_FUTURE"
                    ],
                )
            )

            LEARNED_RESULTS[
                (
                    H,
                    mode,
                    seed
                )
            ] = {
                "model":
                    model,

                "qids":
                    qids,

                "idx":
                    selected,

                "metrics":
                    metrics,

                "best_epoch":
                    best_epoch,

                "best_val":
                    best_val,
            }

if TRAIN_HISTORY:

    history_table = pd.concat(
        TRAIN_HISTORY,
        ignore_index=True,
    )

else:

    history_table = pd.DataFrame()

history_table.to_csv(
    RESULT_DIR /
    "04_training_history.csv",
    index=False,
)


## 22. Per-seed learned results

In [ ]:

seed_rows = []

for H in HORIZONS:

    for mode in [
        "same",
        "cross",
    ]:

        for seed in SEEDS:

            r = LEARNED_RESULTS[
                (
                    H,
                    mode,
                    seed
                )
            ]

            row = {
                "Horizon":
                    H,

                "Mode":
                    mode,

                "Seed":
                    seed,

                "BestEpoch":
                    r[
                        "best_epoch"
                    ],

                "BestValidationMSE":
                    r[
                        "best_val"
                    ],

                "Alpha":
                    float(
                        r[
                            "model"
                        ].alpha.item()
                    ),
            }

            row.update(
                summarize_metrics(
                    r[
                        "metrics"
                    ]
                )
            )

            seed_rows.append(
                row
            )

seed_table = pd.DataFrame(
    seed_rows
)

display(
    seed_table
)

seed_table.to_csv(
    RESULT_DIR /
    "05_seed_results.csv",
    index=False,
)



## 23. Build common-query evaluation for each horizon

SameStock can only be evaluated when the query ticker has at least \(K=10\) historical candidates.

Therefore all Same-vs-Cross comparisons use exactly the same:

\[
\boxed{\text{SameStock-eligible test queries}}
\]

for fairness.


In [ ]:

def seed_average_metrics(
    H,
    mode,
):
    frames = []

    for seed in SEEDS:

        frames.append(
            LEARNED_RESULTS[
                (
                    H,
                    mode,
                    seed
                )
            ][
                "metrics"
            ]
        )

    return pd.DataFrame({
        metric:
            np.stack(
                [
                    x[
                        metric
                    ].to_numpy()
                    for x
                    in frames
                ],
                axis=0,
            ).mean(
                axis=0
            )

        for metric in [
            "AnalogFutureMSE",
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]
    })

HORIZON_QUERY_RESULTS = {}
main_rows = []

for H in HORIZONS:

    sp = SPLITS[
        H
    ]

    same_qids = np.where(
        SAME_PRE[
            H
        ][
            "test_count"
        ] >=
        TOP_K
    )[0]

    same_qids_t = torch.tensor(
        same_qids,
        dtype=torch.long,
        device=DEVICE,
    )

    # Same pattern
    same_pattern_idx = (
        SAME_PRE[
            H
        ][
            "test_idx"
        ][
            same_qids,
            :TOP_K
        ]
    )

    same_pattern_m = metrics_from_indices(
        same_pattern_idx,
        BASIC[
            H
        ][
            "test"
        ][
            "Q_FUTURE"
        ][
            same_qids_t
        ],
        BASIC[
            H
        ][
            "test"
        ][
            "C_FUTURE"
        ],
    )

    # Cross pattern on exactly the same queries
    cross_pattern_idx = (
        CROSS_PRE[
            H
        ][
            "test_idx"
        ][
            same_qids,
            :TOP_K
        ]
    )

    cross_pattern_m = metrics_from_indices(
        cross_pattern_idx,
        BASIC[
            H
        ][
            "test"
        ][
            "Q_FUTURE"
        ][
            same_qids_t
        ],
        BASIC[
            H
        ][
            "test"
        ][
            "C_FUTURE"
        ],
    )

    same_learned_m = (
        seed_average_metrics(
            H,
            "same",
        )
    )

    cross_all_mean = (
        seed_average_metrics(
            H,
            "cross",
        )
    )

    cross_learned_m = (
        cross_all_mean.iloc[
            same_qids
        ]
        .reset_index(
            drop=True
        )
    )

    qmeta = (
        sp[
            "test_query"
        ]
        .iloc[
            same_qids
        ][
            [
                "Ticker",
                "Sector",
                "EndDate",
            ]
        ]
        .reset_index(
            drop=True
        )
    )

    methods = {
        "SameStock Pattern":
            same_pattern_m,

        "SameStock Learned":
            same_learned_m,

        "CrossStock Pattern":
            cross_pattern_m,

        "CrossStock Learned":
            cross_learned_m,
    }

    HORIZON_QUERY_RESULTS[
        H
    ] = {
        "qids":
            same_qids,

        "meta":
            qmeta,

        **methods,
    }

    for method, m in methods.items():

        row = {
            "Horizon":
                H,

            "Method":
                method,

            "NQueries":
                len(
                    m
                ),
        }

        row.update(
            summarize_metrics(
                m
            )
        )

        main_rows.append(
            row
        )

main_table = pd.DataFrame(
    main_rows
)

display(
    main_table
)

main_table.to_csv(
    RESULT_DIR /
    "06_multihorizon_main_common_queries.csv",
    index=False,
)


## 24. CrossStock all-query results

In [ ]:

cross_all_rows = []

for H in HORIZONS:

    # Pattern
    pidx = (
        CROSS_PRE[
            H
        ][
            "test_idx"
        ][
            :,
            :TOP_K
        ]
    )

    pm = metrics_from_indices(
        pidx,
        BASIC[
            H
        ][
            "test"
        ][
            "Q_FUTURE"
        ],
        BASIC[
            H
        ][
            "test"
        ][
            "C_FUTURE"
        ],
    )

    lm = seed_average_metrics(
        H,
        "cross",
    )

    for method, m in [
        (
            "CrossStock Pattern",
            pm,
        ),
        (
            "CrossStock Learned",
            lm,
        ),
    ]:

        row = {
            "Horizon":
                H,

            "Method":
                method,

            "NQueries":
                len(
                    m
                ),
        }

        row.update(
            summarize_metrics(
                m
            )
        )

        cross_all_rows.append(
            row
        )

cross_all_table = pd.DataFrame(
    cross_all_rows
)

display(
    cross_all_table
)

cross_all_table.to_csv(
    RESULT_DIR /
    "07_crossstock_all_query_results.csv",
    index=False,
)


## 25. Improvement summary

In [ ]:

improvement_rows = []

for H in HORIZONS:

    t = (
        main_table[
            main_table[
                "Horizon"
            ] ==
            H
        ]
        .set_index(
            "Method"
        )
    )

    same_pattern = (
        t.loc[
            "SameStock Pattern",
            "ForecastMSE",
        ]
    )

    same_learned = (
        t.loc[
            "SameStock Learned",
            "ForecastMSE",
        ]
    )

    cross_pattern = (
        t.loc[
            "CrossStock Pattern",
            "ForecastMSE",
        ]
    )

    cross_learned = (
        t.loc[
            "CrossStock Learned",
            "ForecastMSE",
        ]
    )

    improvement_rows.extend([
        {
            "Horizon":
                H,

            "Comparison":
                "Same Pattern -> Same Learned",

            "ForecastMSE_Improvement_%":
                100.0 *
                (
                    same_pattern -
                    same_learned
                ) /
                same_pattern,
        },

        {
            "Horizon":
                H,

            "Comparison":
                "Cross Pattern -> Cross Learned",

            "ForecastMSE_Improvement_%":
                100.0 *
                (
                    cross_pattern -
                    cross_learned
                ) /
                cross_pattern,
        },

        {
            "Horizon":
                H,

            "Comparison":
                "Same Learned -> Cross Learned",

            "ForecastMSE_Improvement_%":
                100.0 *
                (
                    same_learned -
                    cross_learned
                ) /
                same_learned,
        },
    ])

improvement_table = pd.DataFrame(
    improvement_rows
)

display(
    improvement_table
)

improvement_table.to_csv(
    RESULT_DIR /
    "08_improvement_summary.csv",
    index=False,
)


## 26. Save query-level paired results

In [ ]:

for H in HORIZONS:

    r = HORIZON_QUERY_RESULTS[
        H
    ]

    q = r[
        "meta"
    ].copy()

    method_key = {
        "SameStock Pattern":
            "SamePattern",

        "SameStock Learned":
            "SameLearned",

        "CrossStock Pattern":
            "CrossPattern",

        "CrossStock Learned":
            "CrossLearned",
    }

    for method, short in (
        method_key.items()
    ):

        m = r[
            method
        ]

        for metric in [
            "AnalogFutureMSE",
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]:

            q[
                f"{short}_{metric}"
            ] = m[
                metric
            ].to_numpy()

    q.to_parquet(
        RESULT_DIR /
        f"09_query_level_H{H}.parquet",
        index=False,
    )


## 27. Moving-block bootstrap helpers

In [ ]:

def date_level_diff(
    dates,
    baseline,
    proposed,
):
    x = pd.DataFrame({
        "Date":
            pd.to_datetime(
                dates
            ),

        "Diff":
            np.asarray(
                baseline
            ) -
            np.asarray(
                proposed
            ),
    })

    return (
        x.groupby(
            "Date"
        )[
            "Diff"
        ]
        .mean()
        .sort_index()
    )


def moving_block_bootstrap(
    date_diff,
    block_len=20,
    n_boot=5000,
    seed=42,
):
    x = date_diff.to_numpy(
        dtype=np.float64
    )

    n = len(
        x
    )

    if n < block_len:
        raise ValueError(
            "Not enough dates."
        )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        pieces = []

        for _ in range(
            n_blocks
        ):

            s = rng.integers(
                0,
                max_start + 1,
            )

            pieces.append(
                x[
                    s:
                    s +
                    block_len
                ]
            )

        sample = (
            np.concatenate(
                pieces
            )[
                :n
            ]
        )

        boot[
            b
        ] = (
            sample.mean()
        )

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_improvement_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),

        "N_dates":
            int(
                n
            ),

        "BlockLength":
            int(
                block_len
            ),
    }


## 28. Bootstrap all main comparisons

In [ ]:

BOOT_COMPARISONS = [
    (
        "SameStock Pattern",
        "SameStock Learned",
        "Same Pattern -> Same Learned",
    ),

    (
        "CrossStock Pattern",
        "CrossStock Learned",
        "Cross Pattern -> Cross Learned",
    ),

    (
        "SameStock Learned",
        "CrossStock Learned",
        "Same Learned -> Cross Learned",
    ),
]

bootstrap_rows = []

for H in HORIZONS:

    r = HORIZON_QUERY_RESULTS[
        H
    ]

    dates = r[
        "meta"
    ][
        "EndDate"
    ]

    for baseline, proposed, label in (
        BOOT_COMPARISONS
    ):

        for metric in [
            "AnalogFutureMSE",
            "ForecastMSE",
            "TerminalMAE",
        ]:

            d = date_level_diff(
                dates,
                r[
                    baseline
                ][
                    metric
                ],
                r[
                    proposed
                ][
                    metric
                ],
            )

            boot = (
                moving_block_bootstrap(
                    d,
                    block_len=BLOCK_LEN,
                    n_boot=N_BOOT,
                    seed=(
                        H *
                        1000 +
                        len(
                            bootstrap_rows
                        )
                    ),
                )
            )

            boot.update({
                "Horizon":
                    H,

                "Comparison":
                    label,

                "Metric":
                    metric,

                "Baseline":
                    baseline,

                "Proposed":
                    proposed,
            })

            bootstrap_rows.append(
                boot
            )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "10_multihorizon_moving_block_bootstrap.csv",
    index=False,
)


## 29. Sector consistency of CrossStock Learned vs Pattern

In [ ]:

sector_rows = []

for H in HORIZONS:

    r = HORIZON_QUERY_RESULTS[
        H
    ]

    tmp = r[
        "meta"
    ][
        [
            "Sector"
        ]
    ].copy()

    tmp[
        "PatternMSE"
    ] = r[
        "CrossStock Pattern"
    ][
        "ForecastMSE"
    ].to_numpy()

    tmp[
        "LearnedMSE"
    ] = r[
        "CrossStock Learned"
    ][
        "ForecastMSE"
    ].to_numpy()

    for sector, g in (
        tmp.groupby(
            "Sector",
            dropna=False,
        )
    ):

        p = g[
            "PatternMSE"
        ].mean()

        l = g[
            "LearnedMSE"
        ].mean()

        sector_rows.append({
            "Horizon":
                H,

            "Sector":
                sector,

            "N":
                len(
                    g
                ),

            "PatternMSE":
                p,

            "LearnedMSE":
                l,

            "Improvement_%":
                100.0 *
                (
                    p -
                    l
                ) /
                p,
        })

sector_table = pd.DataFrame(
    sector_rows
)

display(
    sector_table
)

sector_table.to_csv(
    RESULT_DIR /
    "11_sector_multihorizon.csv",
    index=False,
)


## 30. Final decision summary

In [ ]:

def get_forecast_ci_lower(
    H,
    comparison,
):
    x = bootstrap_table[
        (
            bootstrap_table[
                "Horizon"
            ] ==
            H
        ) &
        (
            bootstrap_table[
                "Comparison"
            ] ==
            comparison
        ) &
        (
            bootstrap_table[
                "Metric"
            ] ==
            "ForecastMSE"
        )
    ]

    return float(
        x[
            "CI_2.5%"
        ].iloc[
            0
        ]
    )

decision_rows = []

for H in HORIZONS:

    t = (
        main_table[
            main_table[
                "Horizon"
            ] ==
            H
        ]
        .set_index(
            "Method"
        )
    )

    same_pattern = t.loc[
        "SameStock Pattern",
        "ForecastMSE",
    ]

    same_learned = t.loc[
        "SameStock Learned",
        "ForecastMSE",
    ]

    cross_pattern = t.loc[
        "CrossStock Pattern",
        "ForecastMSE",
    ]

    cross_learned = t.loc[
        "CrossStock Learned",
        "ForecastMSE",
    ]

    decision_rows.append({
        "Horizon":
            H,

        "CrossLearnedBeatsCrossPattern":
            bool(
                cross_learned <
                cross_pattern
            ),

        "CrossPatternToLearned_Improvement_%":
            100.0 *
            (
                cross_pattern -
                cross_learned
            ) /
            cross_pattern,

        "CrossPatternToLearned_BlockCI_Lower":
            get_forecast_ci_lower(
                H,
                "Cross Pattern -> Cross Learned",
            ),

        "CrossPatternToLearned_Significant":
            bool(
                get_forecast_ci_lower(
                    H,
                    "Cross Pattern -> Cross Learned",
                ) >
                0
            ),

        "SameLearnedBeatsSamePattern":
            bool(
                same_learned <
                same_pattern
            ),

        "SameLearnedToCrossLearned_Improvement_%":
            100.0 *
            (
                same_learned -
                cross_learned
            ) /
            same_learned,

        "SameToCross_BlockCI_Lower":
            get_forecast_ci_lower(
                H,
                "Same Learned -> Cross Learned",
            ),

        "CrossBeatsSameSignificantly":
            bool(
                get_forecast_ci_lower(
                    H,
                    "Same Learned -> Cross Learned",
                ) >
                0
            ),
    })

decision_table = pd.DataFrame(
    decision_rows
)

display(
    decision_table
)

decision_table.to_csv(
    RESULT_DIR /
    "12_final_decision_summary.csv",
    index=False,
)


# Interpretation Guide

Default output directory:

```text
_work/finance_case/results/expanded_multihorizon_final/
```

Key files:

```text
05_seed_results.csv
06_multihorizon_main_common_queries.csv
08_improvement_summary.csv
10_multihorizon_moving_block_bootstrap.csv
11_sector_multihorizon.csv
12_final_decision_summary.csv
```

The primary check is whether CrossStock Learned improves CrossStock Pattern at each horizon with a positive moving-block confidence interval. The SameStock Learned -> CrossStock Learned comparison is secondary and quantifies the additional contribution of cross-security memory. Direction Accuracy is treated as a secondary metric and does not determine the main conclusion.
